# MTM settings recommendation, Phase 1: scoring the recipe grid on NSTX_MTM

Does the extent rule with an envelope margin `s` and a basis floor `F` (the "MTM recipe") reach `m11_new`'s growth-rate accuracy at lower cost, and does the floor fix the shortest-extent cases? Grid: `s` ∈ {1.0, 1.41, 2.0} × NXGRID floor `F` ∈ {14, 18, 22, 26}, `NBASIS = NXGRID − 6`, so the NBASIS floor is `F − 6`. Arms beside the grid: `m3_new`, `m5_new`, `m7_new`, `m9_new`, `m11_new`, and `gftm_fw174` (GFTM default `WIDTH = 1.74`; the NXGRID it actually ran is **24**, per `Fusion_PhD-lgp2`). Reference: GS2 on the NSTX_MTM Latin hypercubes `beta_q_shat_ky_n300` (tuning draw) and `beta_q_shat_ky_n1000` (confirmation draw). The two draws are never pooled.

## Pre-registered selection rule (committed before any cell of the grid was scored)

**Metric.** `medlog` = median over GS2-unstable cases ($\gamma_{GS2} > 10^{-3}\,c_s/a$) of $|\log_{10}(\gamma_{GFTM}/\gamma_{GS2})|$, using GFTM's **dominant** mode (`argmax` of `growth_rate` over `mode`, never `isel(mode=0)`). **Misses are kept, never dropped**: a case where the arm returns no mode with $\gamma > 0$, or has no deck at all (the rule's infeasible cases), scores $+\infty$. Every arm is scored on the same cases. Uncertainty: 95% case-bootstrap intervals (2000 resamples, the same resampled cases for every arm, fixed seed), including the paired difference to `m11_new`; McNemar on tearing-parity capture.

**Rule.** Tune on **n300**. Choose the cheapest `(s, F)` whose n300 medlog is within 0.02 of `m11_new`'s (medlog ≤ medlog(`m11_new`) + 0.02) **and** whose tearing-parity capture is ≥ `m11_new`'s. Report that cell on **n1000** with no re-tuning; if it fails the same test on n1000, that is reported as a failure. *Cheapest* is mean `NBASIS` actually used over the scored cases. *(Agent-added, not the user's: `NBASIS` does not depend on `s` at fixed `F`, so ties in cost are broken by the lower n300 medlog.)*

**Status of the rule in this notebook (user decision, 2026-09-24).** GFTM tearing-parity capture cannot be measured yet: pyrokinetics' `FieldLine.compute_linear_tearing_parameter` needs a theta-resolved `apar`, which a GFTM `gk_output` does not carry (pyrokinetics issue #594, open). The capture half of the rule, and McNemar with it, is **deferred**, so **the recipe is not frozen here**. This notebook reports the accuracy half — which cells come within 0.02 of `m11_new` on n300, and their n1000 values — as **provisional**. GS2's own mode is classified with that diagnostic, which does work on GS2, to split the population into GS2-tearing and other.

**Hypothesis (the user's).** At short extent the rule lacks basis functions. Q1 = cases whose one-sided extent under the `s = 1.41`, `F = 14` arm ($\mathrm{WIDTH}\cdot x_{max}(\mathrm{NXGRID})$) is below the 25th percentile of the GS2-unstable cases, per draw (the `t3ge`/`1c88` definition, a property of the case applied identically to every arm). Verdict, on both draws, at each `s`: **yes** if raising `F` makes the Q1 medlog difference to `m11_new` have a 95% interval that includes or lies below zero, while the Q2–Q4 medlog change from `F = 14` to `F = 26` has an interval that includes zero; **partial** if the Q1 gap narrows by at least half its `F = 14` value but one of those conditions fails; **no** otherwise.
